In [0]:
# COMMAND ----------
# DBTITLE 1,Importações e Parâmetros
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

CATALOGO = "bradesco_prod"
TABELA_SILVER = f"{CATALOGO}.crm_silver.opportunity_clean"
TABELA_GOLD = f"{CATALOGO}.crm_gold.oportunidades_por_estagio"

# COMMAND ----------
# DBTITLE 2,Leitura da Camada Silver
df_silver = spark.table(TABELA_SILVER)

# COMMAND ----------
# DBTITLE 3,Agregações e Regras de Negócio para BI
df_gold = (
    df_silver
    .groupBy("StageName")
    .agg(
        {"Amount": "sum", "Id": "count"}
    )
    .withColumnRenamed("sum(Amount)", "valor_total_oportunidades")
    .withColumnRenamed("count(Id)", "qtd_oportunidades")
)

# COMMAND ----------
# DBTITLE 4,Gravação na Camada Gold
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABELA_GOLD)
)

print(f"Tabela Gold consolidada com sucesso: {TABELA_GOLD}")